In [1]:
import numpy as np
from helpers import *
from implementations import *

Import the data

In [2]:
# You have to change the path for it to work
data_path = r'C:\Users\natha\Documents\EPFL\Cours_MA1\ML\ML_course\projects\project1 - withGit\data\dataset'

In [3]:
x_train, x_test, y_train, train_ids, test_ids, headers_train = load_csv_data(data_path, sub_sample=False)

test


In [4]:
print(x_train.shape)
print(x_test.shape)
print(y_train.shape)

(328135, 321)
(109379, 321)
(328135,)


Convert -1 into 0 in y

In [5]:
y_train[y_train == -1] = 0

# Check if all values are either 0 or 1
if np.all((y_train == 0) | (y_train == 1)):
    print("All values are either 0 or 1.")
else:
    raise ValueError("The array contains values other than 0 or 1.")

All values are either 0 or 1.


Remove columns with Nan

In [6]:
# Function to find the count of missing values in each columns
def find_missing_values(data, headers=None):
    num_rows, num_cols = data.shape
    missing_count = np.zeros(num_cols, dtype=int)  
    columns = np.linspace(0,num_cols, num_cols+1)
    
    for col in range(num_cols):
        # Count the missing values
        missing_count[col] = np.sum(np.isnan(data[:, col]))
    if headers : 
        # Returning only the columns with missing values
        missing_info = {headers[col]: missing_count[col] for col in range(num_cols) if missing_count[col] > 0}
    else : 
        missing_info = {columns[col]: missing_count[col] for col in range(num_cols) if missing_count[col] > 0}
    return missing_info

In [7]:
# Function to remove columns with more than 1/3 of missing values
def remove_high_missing_columns(data):
    num_rows, num_cols = data.shape
    threshold = 0
    missing_count = find_missing_values(data) 

    # Create a mask for columns to keep
    columns_to_keep = [col for col in range(num_cols) if missing_count.get(col, 0) <= threshold]

    # Return a new array with only the columns that meet the criteria
    return data[:, columns_to_keep]

In [8]:
sliced_x_train = remove_high_missing_columns(x_train)

In [9]:
sliced_x_train.shape

(328135, 82)

In [10]:
column_with_nan = find_missing_values(sliced_x_train)

In [11]:
print(column_with_nan)

{}


Standardize data to avoid issues with large numbers

In [12]:
def standardize(x):
    """Stadartize the input data x

    Args:
        x: numpy array of shape=(num_samples, num_features)

    Returns:
        standartized data, shape=(num_samples, num_features)

    >>> standardize(np.array([[1, 2], [3, 4], [5, 6]]))
    array([[-1.22474487, -1.22474487],
           [ 0.        ,  0.        ],
           [ 1.22474487,  1.22474487]])
    """
    # ***************************************************
    means=np.mean(x, axis=0)
    stds=np.std(x, axis=0)
    std_data=(x-means)/stds
    # ***************************************************

    return std_data

In [13]:
x_train = standardize(sliced_x_train)

Split the data

In [14]:
def split_data(x, y, ratio, seed=1):
    """
    split the dataset based on the split ratio. If ratio is 0.8
    you will have 80% of your data set dedicated to training
    and the rest dedicated to testing. If ratio times the number of samples is not round
    you can use np.floor. Also check the documentation for np.random.permutation,
    it could be useful.

    Args:
        x: numpy array of shape (N,), N is the number of samples.
        y: numpy array of shape (N,).
        ratio: scalar in [0,1]
        seed: integer.

    Returns:
        x_tr: numpy array containing the train data.
        x_te: numpy array containing the test data.
        y_tr: numpy array containing the train labels.
        y_te: numpy array containing the test labels.

    >>> split_data(np.arange(13), np.arange(13), 0.8, 1)
    (array([ 2,  3,  4, 10,  1,  6,  0,  7, 12,  9]), array([ 8, 11,  5]), array([ 2,  3,  4, 10,  1,  6,  0,  7, 12,  9]), array([ 8, 11,  5]))
    """
    # set seed
    np.random.seed(seed)
    # ***************************************************
    indices = np.random.permutation(x.shape[0])
    x_shuffled = x[indices]
    y_shuffled = y[indices]
    
    first_index_te = int(np.floor(x.shape[0] * ratio))
    x_tr = x_shuffled[:first_index_te]
    x_te = x_shuffled[first_index_te:]
    y_tr = y_shuffled[:first_index_te]
    y_te = y_shuffled[first_index_te:]
    return x_tr, x_te, y_tr, y_te
    # ***************************************************

In [15]:
x_tr, x_val, y_tr, y_val = split_data(x_train, y_train, 0.8, seed=1)
print(x_tr.shape)
print(x_val.shape)
print(y_tr.shape)
print(y_val.shape)

(262508, 82)
(65627, 82)
(262508,)
(65627,)


Train the model

In [17]:
lambda_ = 0.1
initial_w = np.zeros(x_tr.shape[1])
max_iters = 1000
gamma = 0.01

w_opt, loss_opt = reg_logistic_regression(y_tr, x_tr, lambda_, initial_w, max_iters, gamma)

Current iteration=0, loss=0.6931471805599448
Current iteration=100, loss=0.6804435130428013
Current iteration=200, loss=0.6780428734603308
Current iteration=300, loss=0.6771173833331848
Current iteration=400, loss=0.6766637288087088
Current iteration=500, loss=0.6764158700185852
Current iteration=600, loss=0.6762718486275859
Current iteration=700, loss=0.6761847957244556
Current iteration=800, loss=0.6761307062505011
Current iteration=900, loss=0.6760964001495705
loss=0.6760742856974865


In [18]:
print(w_opt)

[ 1.39766801e-03  1.46922803e-03  2.38431202e-04  3.40034449e-04
 -4.26787032e-03 -4.00452982e-03 -1.43193635e-03 -2.81443536e-03
 -2.81443536e-03 -1.45161644e-02 -6.14535744e-03 -2.50378588e-03
 -8.56095299e-03 -4.88640799e-02 -1.11451321e-02 -1.30028274e-02
 -3.46778044e-02 -1.59147441e-02 -2.31839428e-02 -6.08170708e-02
 -4.12481551e-03 -1.77142183e-02  9.53274701e-03  5.56739502e-02
 -3.42921929e-03  1.39059181e-03 -3.71241016e-03 -1.09078055e-03
  7.94522371e-03 -2.84220568e-03  7.94717138e-02  3.33010888e-02
  6.18704612e-02 -1.17801331e-02  1.48513574e-02  1.16730679e-02
 -1.29866493e-02 -2.49567496e-03 -1.79914616e-03  2.15440140e-03
 -1.40731350e-03  2.64605954e-04 -1.22123946e-03  4.09648759e-02
  2.97500611e-02  4.31816524e-02  2.75318064e-02 -3.59976120e-03
 -7.20005876e-03 -1.52754374e-02 -1.98110920e-02 -2.14523490e-02
  5.83397438e-03  1.44708899e-02 -5.07374634e-03 -3.39738783e-03
 -2.80486822e-04 -1.84233837e-03 -1.10963501e-03 -1.72228717e-03
  3.00204108e-04  1.39615

Apply the model to the validation set

In [44]:
g = x_val @ w_opt
g.shape

(65627,)

In [45]:
limit = 0.5

g[g <= limit] = 0
g[g > limit] = 1

# Check if all values are either 0 or 1
if np.all((g == 0) | (g == 1)):
    print("All values are either 0 or 1.")
else:
    raise ValueError("The array contains values other than 0 or 1.")

All values are either 0 or 1.


Check our accuracy and F1

In [46]:
accuracy = np.mean(y_val == g)

print("Accuracy:", accuracy)

Accuracy: 0.902936291465403


In [47]:
TP = np.sum((y_val == 1) & (g == 1))
FP = np.sum((y_val == 0) & (g == 1))
FN = np.sum((y_val == 1) & (g == 0))

# Calculate Precision and Recall
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0

# Calculate F1 Score
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1_score)

Precision: 0.39018328673501085
Recall: 0.22179057036906233
F1 Score: 0.2828191848682729
